# Defect Measurement Pipeline

Measure physical size of **scratch_dent** and **scratch_line** detected by the trained YOLO-seg model.

Per defect mask it computes:
- **length** = major axis (longest span)
- **width**  = minor axis (perpendicular)
- **area**   = true mask pixel area

Pixel → physical via `UM_PER_PIXEL` (calibrate it).

Run cells top to bottom. Edit **Config** then run **Run measurement**.

## 1. Install (run once)

In [15]:
# run once if any import fails — installs into THIS kernel
%pip install pandas matplotlib opencv-python ultralytics pyyaml

## 2. Config — edit these

In [ ]:
from pathlib import Path

# path to the ML_pipeline folder (holds trained model). sibling of this folder.
ML_PIPELINE_DIR = (Path.cwd().parent / 'ML_pipeline').resolve()

MODEL_PATH = ML_PIPELINE_DIR / 'models' / 'train' / 'weights' / 'best.pt'
INPUT_DIR  = ML_PIPELINE_DIR / 'test_result'        # images to measure
OUTPUT_DIR = Path('./output').resolve()             # results land here

# --- CALIBRATION: real microns per 1 pixel. CHANGE THIS to your real value ---
UM_PER_PIXEL = 1.0

CONF      = 0.25                 # min detection confidence
CLASS_IDS = [3, 4]              # scratch_dent=3, scratch_line=4
CLASS_NAMES = {0: 'comtam_dust', 1: 'comtam_stain', 2: 'comtam_x',
               3: 'scratch_dent', 4: 'scratch_line', 5: 'other'}

OUTPUT_DIR.mkdir(exist_ok=True)
assert MODEL_PATH.exists(), f'model not found: {MODEL_PATH}'
assert INPUT_DIR.exists(),  f'input not found: {INPUT_DIR}'
print('model :', MODEL_PATH)
print('input :', INPUT_DIR)
print('scale :', UM_PER_PIXEL, 'um/px  =', UM_PER_PIXEL/1000, 'mm/px')

## 3. Imports + measurement helper

In [17]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO


def mask_measures(mask_uint8):
    """Return (length_px, width_px, area_px) from a binary mask.
    length/width = oriented min-area box max/min side. area = real pixel count."""
    area_px = float(cv2.countNonZero(mask_uint8))
    if area_px == 0:
        return 0.0, 0.0, 0.0
    cnts, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return 0.0, 0.0, area_px
    cnt = max(cnts, key=cv2.contourArea)
    (_, _), (w, h), _ = cv2.minAreaRect(cnt)
    return float(max(w, h)), float(min(w, h)), area_px

ModuleNotFoundError: No module named 'pandas'

## 4. Load model

In [ ]:
model = YOLO(str(MODEL_PATH))
print('loaded:', MODEL_PATH.name)

## 5. Run measurement
Set `SAVE_ANNOTATED=True` to also write annotated PNGs to `output/`.

In [ ]:
SAVE_ANNOTATED = True

want = set(CLASS_IDS)
imgs = sorted(INPUT_DIR.glob('*.png')) + sorted(INPUT_DIR.glob('*.jpg'))
rows = []
print(f'{len(imgs)} images...')

for img_path in imgs:
    res = model.predict(source=str(img_path), conf=CONF, verbose=False)[0]
    if res.masks is None:
        continue
    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]
    polys = res.masks.xy
    clss = res.boxes.cls.cpu().numpy().astype(int)
    confs = res.boxes.conf.cpu().numpy()

    n = 0
    for i, poly in enumerate(polys):
        cid = int(clss[i])
        if cid not in want:
            continue
        mask = np.zeros((H, W), np.uint8)
        cv2.fillPoly(mask, [poly.astype(np.int32)], 255)
        length_px, width_px, area_px = mask_measures(mask)
        s = UM_PER_PIXEL
        rows.append({
            'image': img_path.name, 'defect_id': n,
            'class': CLASS_NAMES.get(cid, cid), 'conf': round(float(confs[i]), 3),
            'length_px': round(length_px, 1), 'width_px': round(width_px, 1),
            'area_px': round(area_px, 1),
            'length_um': round(length_px*s, 1), 'width_um': round(width_px*s, 1),
            'area_um2': round(area_px*s*s, 1),
            'length_mm': round(length_px*s/1000, 4), 'width_mm': round(width_px*s/1000, 4),
            'area_mm2': round(area_px*s*s/1e6, 6),
        })
        if SAVE_ANNOTATED:
            color = (0, 0, 255) if CLASS_NAMES.get(cid) == 'scratch_line' else (0, 165, 255)
            cv2.polylines(img, [poly.astype(np.int32)], True, color, 2)
            x, y = poly[0].astype(int)
            cv2.putText(img, f"{CLASS_NAMES.get(cid, cid)} L{length_px*s:.0f} W{width_px*s:.0f}um",
                        (int(x), max(15, int(y)-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        n += 1
    if SAVE_ANNOTATED and n > 0:
        cv2.imwrite(str(OUTPUT_DIR / img_path.name), img)

df = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / 'measurements.csv'
df.to_csv(csv_path, index=False)
print(f'{len(df)} defects measured -> {csv_path}')

## 6. Inspect results table

In [ ]:
df.head(20)

## 7. Summary per class

In [ ]:
if len(df):
    display(df.groupby('class').agg(
        count=('length_um', 'size'),
        mean_length_um=('length_um', 'mean'),
        max_length_um=('length_um', 'max'),
        mean_width_um=('width_um', 'mean'),
        mean_area_um2=('area_um2', 'mean'),
    ).round(1))
else:
    print('no defects')

## 8. Preview annotated images

In [ ]:
import matplotlib.pyplot as plt

shown = sorted(OUTPUT_DIR.glob('*.png'))[:6]
if shown:
    cols = 3
    rows_n = (len(shown) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(15, 5*rows_n))
    for ax, p in zip(np.array(axes).ravel(), shown):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(p.name, fontsize=8)
        ax.axis('off')
    for ax in np.array(axes).ravel()[len(shown):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('no annotated images (set SAVE_ANNOTATED=True and rerun cell 5)')

## 9. Measure a single image (quick test)

In [ ]:
TEST_IMG = imgs[0] if imgs else None   # or set Path('path/to/img.png')

if TEST_IMG:
    res = model.predict(source=str(TEST_IMG), conf=CONF, verbose=False)[0]
    img = cv2.imread(str(TEST_IMG)); H, W = img.shape[:2]
    if res.masks is not None:
        for i, poly in enumerate(res.masks.xy):
            cid = int(res.boxes.cls[i])
            if cid not in want:
                continue
            m = np.zeros((H, W), np.uint8); cv2.fillPoly(m, [poly.astype(np.int32)], 255)
            L, Wd, A = mask_measures(m)
            print(f"{CLASS_NAMES.get(cid)}: length={L*UM_PER_PIXEL:.1f}um width={Wd*UM_PER_PIXEL:.1f}um area={A*UM_PER_PIXEL**2:.1f}um2")
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()